## Customer Order Data Display

In [ ]:
%%capture cap --no-display

%load_ext autoreload
%autoreload 2

# --- IMPORTS ---
import importlib, html
from datetime import datetime
from collections import defaultdict
from app import create_app, db
from models import Customer
import plotly.graph_objects as go
import plotly.io as pio

# --- CONFIG ---
# Set this based on your Customer model:
#   True  => model has first_name + last_name
#   False => model has full_name
HAS_SEPARATE_NAMES = True  # change to False if you use full_name


# Helper to convert "$xx.yy" -> float
def money_to_float(v):
    if v is None:
        return 0.0
    # Accept values already numeric or "$" prefixed strings
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v)
    s = re.sub(r"[^0-9.\-]", "", s)  # strip $ and commas
    try:
        return float(s) if s else 0.0
    except ValueError:
        return 0.0

# --- QUERY ---
app = create_app()
with app.app_context():
    customers = Customer.query.order_by(Customer.created_at.asc()).all()

# Helper to get display name depending on schema
def get_name(c):
    if HAS_SEPARATE_NAMES:
        return f"{getattr(c,'first_name','').strip()} {getattr(c,'last_name','').strip()}".strip()
    return getattr(c, "full_name", "").strip()

# --- Store customer orders ---
# 1) Customer list (all names, including duplicates)
customers_list = [get_name(c) for c in customers]

# 2) Customer set (unique names)
customers_set = set(customers_list)

# 3) Customer tuples (detailed)
def row_tuple(c):
    name = get_name(c)
    base = (name, getattr(c, "category", None), c.price)
    # (Name, Product, Category, Email, Phone, Price)
    return (name,  getattr(c,"product",None), *base[1:])

customer_details_tuples = [row_tuple(c) for c in customers]

# Figure out tuple headers (aligned with row_tuple)
tuple_headers = ["Customer Name", "Product", "Category", "Price"]

# 4) Customer dictionary (name -> product column)

customers_dict_list = defaultdict(list)

for c in customers:
  name = get_name(c)
  product = c.product
  if product and product not in customers_dict_list[name]:
    customers_dict_list[name].append(product)

customers_dict = {name: ", ".join(values) for name, values in customers_dict_list.items()}

# ------------- Classify products by category-------------------

# 5) Product Category Dictionary  (Product → Category)
product_category_dict = defaultdict(list)
for c in customers:
  product = getattr(c, "product", None)
  category = getattr(c, "category", None)
  if c.product and c.category not in product_category_dict[c.product]:
    product_category_dict[c.product].append(c.category)

product_category_dict = {product: ", ".join(categories) for product, categories in product_category_dict.items()}

# 6) Unique Products Categories (for reference)
unique_product_categories = set()
for c in customers:
    category = getattr(c, "category", None)
    if category not in unique_product_categories:
        unique_product_categories.add(category)

# 7) Category → [Products] dictionary
category_products_dict = defaultdict(set)
for c in customers:
    category = getattr(c, "category", None)
    product = getattr(c, "product", None)
    if category and product not in category_products_dict[category]:
        category_products_dict[category].add(product)
category_products_dict = {cat: ", ".join(sorted(prods)) for cat, prods in category_products_dict.items()}

# --------------Analyze customer orders--------------------

# 8) Custome spending dictionary (name → total spent)
customer_spending = defaultdict(float)
for c in customers:
  name = get_name(c)
  price = getattr(c, "price", 0.0)
  if price:
    customer_spending[name] += price
customer_spending = {name: f"${total:.2f}" for name, total in customer_spending.items()}

# ---- Customer Spending Bar (descending) ----
cust_names = list(customer_spending.keys())
cust_totals = [money_to_float(v) for v in customer_spending.values()]

# Sort descending for nicer chart
cust_sorted = sorted(zip(cust_names, cust_totals), key=lambda x: x[1], reverse=True)
cust_names_sorted, cust_totals_sorted = zip(*cust_sorted) if cust_sorted else ([], [])

fig_customer_spending = go.Figure(
    data=[
        go.Bar(
            x=cust_names_sorted,
            y=cust_totals_sorted,
            marker=dict(color="#4F46E5")  # indigo
        )
    ]
)
fig_customer_spending.update_layout(
    title="Total Spending by Customer",
    xaxis_title="Customer",
    yaxis_title="Total Spent (USD)",
    margin=dict(l=40, r=20, t=60, b=80),
    height=420
)

# 9) Create a dictionary of customer based on the total expenditure, sorted in descending order. 
# if expenditure is above $100, mark them as "High Value", between $50 and $100 as "Medium Value", and below $50 as "Low Value".
customer_value_dict = defaultdict(str)
for name, total in customer_spending.items():
    value = "Low Value"
    if float(total.strip("$")) > 100:
        value = "High Value"
    elif float(total.strip("$")) >= 50:
        value = "Medium Value"
    customer_value_dict[name] = value

#---------------Generate business insights-------------------

# 10) Calculate the total revenute generated by each product category.
product_category_revenue = defaultdict(float)
for c in customers:
    category = getattr(c, "category", None)
    price = getattr(c, "price", 0.0)
    if category and price:
        product_category_revenue[category] += price
product_category_revenue = {cat: f"${total:.2f}" for cat, total in product_category_revenue.items()}


# ---- Category Revenue Bar (descending) ----
cat_names = list(product_category_revenue.keys())
cat_totals = [money_to_float(v) for v in product_category_revenue.values()]

cat_sorted = sorted(zip(cat_names, cat_totals), key=lambda x: x[1], reverse=True)
cat_names_sorted, cat_totals_sorted = zip(*cat_sorted) if cat_sorted else ([], [])

fig_category_revenue = go.Figure(
    data=[
        go.Bar(
            x=cat_names_sorted,
            y=cat_totals_sorted,
            marker=dict(color="#10B981")  # emerald
        )
    ]
)
fig_category_revenue.update_layout(
    title="Total Revenue by Category",
    xaxis_title="Category",
    yaxis_title="Revenue (USD)",
    margin=dict(l=40, r=20, t=60, b=80),
    height=420
)

#11) All produucts from all categories (for reference)
all_products = set()
for p in category_products_dict.values():
   for product in p.split(", "):
       all_products.add(product)

#12) All Customers who purchased from a specific category (e.g., "Electronics").
electronics_customers = {get_name(c) for c in customers if getattr(c, "category", None) == "Electronics"}

#13) Top 3 customers by spending
top_customers = sorted(customer_spending.items(), key=lambda x: float(x[1].strip("$")), reverse=True)[:3]

#--------------------Organize and display data-------------------

#14) print customer total spending as  per category in a table format.
customer_category_spending = defaultdict(lambda: defaultdict(float))
for c in customers:
    name = get_name(c)
    category = getattr(c, "category", None)
    price = getattr(c, "price", 0.0)
    if name and category and price:
        customer_category_spending[name][category] += price

#15) Identify customers who have purchased from multiple categories
customer_multi_category = set()
for name, cats in customer_category_spending.items():
    if len(cats) > 1:
        customer_multi_category.add(name)

#16) Identify the customers who have purchased from electronics and clothing category
electronics_and_clothing_customers = set()
for name, cats in customer_category_spending.items():
    if "Electronics" in cats and "Clothing" in cats:
        electronics_and_clothing_customers.add(name)


# Convert to embeddable HTML snippets (no full HTML, Plotly JS via CDN)
customer_spending_html = pio.to_html(fig_customer_spending, full_html=False, include_plotlyjs='cdn')
category_revenue_html  = pio.to_html(fig_category_revenue, full_html=False, include_plotlyjs=False)

# --- RENDER HELPERS ---
def html_escape(s):
    return html.escape("" if s is None else str(s))

def render_list_as_ul(items):
    return "<ul>\n" + "\n".join(f"  <li>{html_escape(x)}</li>" for x in items) + "\n</ul>"

def render_set_as_ul(items_set):
    # sort for stable output
    items = sorted(items_set)
    return render_list_as_ul(items)

def render_dict_as_table(d, key_label='Key', value_label='Value'):
      rows = "".join(
            f"<tr><td>{html_escape(k)}</td><td>{html_escape(v)}</td></tr>"
            for k, v in d.items()
      )
      return f"""<table class="kv">
   <thead><tr><th>{html_escape(key_label)}</th><th>{html_escape(value_label)}</th></tr></thead>
   <tbody>
      {rows}
   </tbody>
</table>
"""

def render_tuples_as_table(headers, rows):
    thead = "".join(f"<th>{html_escape(h)}</th>" for h in headers)
    body_rows = []
    for r in rows:
        tds = "".join(f"<td>{html_escape(x)}</td>" for x in r)
        body_rows.append(f"<tr>{tds}</tr>")
    return f"""
<table class="grid">
  <thead><tr>{thead}</tr></thead>
  <tbody>
    {"".join(body_rows)}
  </tbody>
</table>
"""

# --- PAGE STYLES ---
CSS = """
<style>
  body { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; margin: 24px; color: #222; }
  h1 { margin: 0 0 4px 0; }
  h2 { margin: 24px 0 8px 0; }
  .muted { color: #666; }
  .section { margin-bottom: 28px; }
  .scroll { max-height: 420px; overflow: auto; border: 1px solid #e5e7eb; padding: 12px; border-radius: 8px; background: #fafafa; }
  table { border-collapse: collapse; width: 100%; }
  table.grid th, table.grid td, table.kv th, table.kv td { border: 1px solid #e5e7eb; padding: 8px; }
  table.grid th { background: #f3f4f6; text-align: left; }
  table.kv th { background: #f9fafb; text-align: left; }
  code.kv { background: #f6f8fa; padding: 2px 6px; border-radius: 4px; }
  .pill { display: inline-block; padding: 2px 8px; background: #eef2ff; color: #3730a3; border-radius: 999px; font-size: 12px; }
  .row { display: grid; grid-template-columns: 1fr; gap: 16px; }
  @media (min-width: 900px) { .row { grid-template-columns: 1fr 1fr; } }
</style>
"""

# --- CAPTURED STDOUT ---
captured_stdout_html = html.escape(cap.stdout)

# --- COMPOSE HTML ---
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

html_report = f"""<!doctype html>
<html lang="en">
   <head>
      <meta charset="utf-8" />
      <title>Customer Insights Report</title>
      {CSS}
   </head>
   <body>
      <h1>Customer Insights Report</h1>
      <div class="muted">Generated at: {html_escape(generated_at)}</div>
      <div class="section">
         <h2>1) Customer Tuples <span class="pill">Detailed rows</span></h2>
         <div class="scroll">
            {render_tuples_as_table(tuple_headers, customer_details_tuples)}
         </div>
         <div class="muted">Showing {len(customer_details_tuples)} rows.</div>
      </div>
      <div class="row">
         <div class="section">
            <h2>2) Customer List <span class="pill">Duplicates allowed</span></h2>
            <div class="scroll">
               {render_list_as_ul(customers_list)}
            </div>
            <div class="muted">List length: <code class="kv">{len(customers_list)}</code></div>
         </div>
         <div class="section">
            <h2>3) Customer Set <span class="pill">Unique names</span></h2>
            <div class="scroll">
               {render_set_as_ul(customers_set)}
            </div>
            <div class="muted">Unique names: <code class="kv">{len(customers_set)}</code></div>
         </div>
      </div>
      <div class="section">
         <h2>4) Customer Dictionary <span class="pill">Name → {'Product'}</span></h2>
         <div class="scroll">
            {render_dict_as_table(customers_dict, 'Customer Name', 'Product')}
         </div>
         <div class="muted">Pairs: <code class="kv">{len(customers_dict)}</code></div>
      </div>
      <div class="section">
         <h2>5) Product Category Dictionary <span class="pill">Product → Category</span></h2>
         <div class="scroll">
            {render_dict_as_table(product_category_dict, 'Product', 'Category')}
         </div>
         <div class="muted">Pairs: <code class="kv">{len(product_category_dict)}</code></div>
      </div>
      <div class="section">
         <h2>6) Unique Product Categories <span class="pill">List of categories</span></h2>
         <div class="scroll">
            {render_set_as_ul(unique_product_categories)}
         </div>
         <div class="muted">Count: <code class="kv">{len(unique_product_categories)}</code></div>
      </div>
      <div class="section">
      <h2>7) Category → Products Dictionary <span class="pill">Category → [Products]</span></h2>
      <div class="scroll">
         {render_dict_as_table(category_products_dict, 'Category', 'Products')}
      </div>
      <div class="muted">Pairs: <code class="kv">{len(category_products_dict)}</code></div>
      <div class="section">
      <h2>8) Customer Spending <span class="pill">Total spent per customer</span></h2>
      <div>
        {customer_spending_html}
      </div>
      <div class="scroll">
         {render_dict_as_table(customer_spending, 'Customer Name', 'Total Spent')}
      </div>
      <div class="muted">Customers: <code class="kv">{len(customer_spending)}</code></div>
      <div class="section">
      <h2>9) Customer Value Segment <span class="pill">High/Medium/Low Value</span></h2>
      <div class="scroll">
         {render_dict_as_table(customer_value_dict, 'Customer Name', 'Value Segment')}
      </div>
      <div class="muted">Customers: <code class="kv">{len(customer_value_dict)}</code></div>
      <div class="section">
      <h2>10) Product Category Revenue <span class="pill">Total revenue per product category</span></h2>    
      <div>
        {category_revenue_html}
       </div>
      <div class="scroll">
         {render_dict_as_table(product_category_revenue, 'Category', 'Revenue')}
      </div>
      <div class="muted">Categories: <code class="kv">{len(product_category_revenue)}</code></div>
      <div class="section">
      <h2>11) All Products List <span class="pill">Unique products across all categories</span></h2>
      <div class="scroll">
         {render_set_as_ul(all_products)}
      </div>
      <div class="muted">Products: <code class="kv">{len(all_products)}</code></div>
      <div class="section">
      <h2>12) Electronics Customers <span class="pill">Customers who purchased Electronics</span></h2>
      <div class="scroll">
         {render_list_as_ul(electronics_customers)}
      </div>
       <div class="muted">Customers: <code class="kv">{len(electronics_customers)}</code></div>
       <div class="section">
      <h2>13) Top 3 Customers by Spending <span class="pill">Top Spenders</span></h2>
      <div class="scroll">
        {render_tuples_as_table(['Customer Name', 'Total Spent'], top_customers)}
      </div>
      <div class="muted">Showing top 3 customers.</div>
      <div class="section">
      <h2>14) Customer Spending by Category <span class="pill">Spending breakdown</span></h2>
      <div class="scroll">
        {render_dict_as_table({name: ", ".join(f"{cat}: ${amt:.2f}" for cat, amt in cats.items()) for name, cats in customer_category_spending.items()}, 'Customer Name', 'Spending by Category')}
      </div>
      <div class="muted">Customers: <code class="kv">{len(customer_category_spending)}</code></div>
      <div class="section">
      <h2>15) Multi-Category Customers <span class="pill">Customers who purchased from multiple categories</span></h2>
      <div class="scroll">
        {render_set_as_ul(customer_multi_category)}
      </div>
      <div class="muted">Customers: <code class="kv">{len(customer_multi_category)}</code></div>
      <div class="section">
      <h2>16) Electronics & Clothing Customers <span class="pill">Customers who purchased from both Electronics and Clothing</span></h2>
      <div class="scroll">
        {render_set_as_ul(electronics_and_clothing_customers)}
      </div>
      <div class="muted">Customers: <code class="kv">{len(electronics_and_clothing_customers)}</code></div>
      <div class="section">
         <h2>Raw Data From Customers Database</h2>
         <div class="scroll">
            <pre class="prewrap">{captured_stdout_html}</pre>
         </div>
      </div>
   </body>
</html>
"""

# --- WRITE FILE ---
out_path = "customer_report.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html_report)

print(f"Wrote: {out_path}")

In [ ]:
import webbrowser, os
webbrowser.open('file://' + os.path.abspath('customer_report.html'))